In [ ]:
import numpy as np
from gould_2026.datasets import Zong22Dataset
from sim_stim import make_srs, get_sim_stim_preset
import matplotlib.pyplot as plt
from gould_2026.stim_designer import StimDesigner, OptimizationMethod
from gould_2026.plotting import Palette
from itertools import cycle

In [ ]:
output = None
condition = 'constrained'

In [ ]:
if condition == 'constrained':
    opt_method, scale_factor = OptimizationMethod.JAXOPT, 2
elif condition == 'positive_constrained':
    opt_method, scale_factor = OptimizationMethod.JAXOPT_POSITIVE_CONSTRAINED, 2
elif condition == 'sparse_constrained':
    opt_method, scale_factor = OptimizationMethod.JAXOPT_SPARSE_CONSTRAINED, 2
elif condition == 'unconstrained':
    opt_method, scale_factor = OptimizationMethod.JAXOPT_UNCONSTRAINED, .5
else:
    raise ValueError()


In [ ]:

from gould_2026.save_to_cache import save_to_cache
@save_to_cache('starburst_setup', location='/mnt/data/gould_2026_cache/')
def to_cache():
    rng = np.random.default_rng(0)
    d = Zong22Dataset()
    data = d.neural_data

    to_run = {
        'learning from stim': dict(stim_magnitude=10, optimization_method=OptimizationMethod.JAXOPT, u_to_s_model_type='identity', exit_time=np.inf, stim_rate=None, smoothing_tau=1, centerer_init_size=8 * 25, initial_nostim_period=30, regular_stim_iter=cycle([1 / 10, 1 / 3]), stim_timing_method='regular', attempt_correction=True, heed_stimuli=True),
    }

    srs = make_srs(data, rng, to_run=to_run, n_runs=1, show_tqdm=True)

    sr = srs['learning from stim'][0]
    return sr

sr = to_cache()

In [ ]:
fig, axs = plt.subplots(figsize=(5,5), layout='constrained', squeeze=False)


stim_designer = StimDesigner(should_log=True, rng_seed=1)

theta=.1
i=40
l=1
r=0

latents = sr.log['latents'].slice_by_time(slice(30,None))
ax = axs[0,0]
ax.plot(latents[:, 0], latents[:, 1], alpha=.1, color='k')

center_t = sr.log['stim_intended_samples'].t[i]
latents = sr.log['latents'].slice_by_time(slice(center_t-l,center_t+r))
line = ax.plot(latents[:-1, 0], latents[:-1, 1], color='C0')
stim_s = sr.log['stim_intended_samples'].slice_by_time(slice(center_t-l,center_t+r)).t - latents.dt
latents_s = latents.slice_by_time(stim_s).reshape((-1, latents.shape[1]))
ax.plot(latents_s[:, 0], latents_s[:, 1], '.', color='g')



u = sr.stim_designer.log[i]['u']
v = sr.stim_designer.log[i]['v']

for theta in np.linspace(0, 2*np.pi, 20) - 0.11:
    v = 0 * v
    v[0,0] = np.cos(theta)
    v[1,0] = np.sin(theta)

    equivalent_projection_matrix = sr.stim_designer.log[i]['equiv_proj_mat']
    u_to_s_function=lambda u: equivalent_projection_matrix.T @ u

    stim_designer.optimization_method = opt_method
    new_u = stim_designer.design_stim(v=v, u_dimension=u.size, u_to_s_function=u_to_s_function, equivalent_projection_matrix=equivalent_projection_matrix)

    s = u_to_s_function(new_u) * scale_factor
    axs[0,0].annotate(
        '',
        xytext=(latents_s[0,0], latents_s[0,1]),
        xy=(latents_s[0,0]+s[0], latents_s[0,1]+s[1]),
        arrowprops=dict(color=Palette.s_designed, width=1.5),
        size=20,
        annotation_clip=False
    )


axs[0,0].axis('equal')
axs[0,0].axis('off')
axs[0,0].set_xlim(np.array([-2,2]) + 1.26)
axs[0,0].set_ylim(np.array([-2,2]) + 0.73)


if output is not None:
    fig.savefig(output, dpi=400)